In [4]:
import json

import torch
import gc
from sentence_transformers import SentenceTransformer
import pandas as pd
import ipywidgets as w
from pathlib import Path

In [5]:
title_text_pairs = pd.read_parquet("articles_10k_test.parquet")[:]

texts = title_text_pairs["text"].tolist()
titles = title_text_pairs["section_title"].tolist()


def truncate_text(text, max_length=200):
    if len(text) > max_length:
        return text[:max_length] + '...'
    return text


def find_best_title(index_in_titles: int, k=5, print_result: bool = True):
    text_embedding = text_embeddings[index_in_titles]
    similarities: torch.Tensor = model.similarity(text_embedding, title_embeddings)

    best_indices = similarities.topk(k=k, largest=True).indices.cpu().numpy().astype(int).flatten()

    output = None

    text = texts[index_in_titles] if index_in_titles < len(texts) else None
    true_title = titles[index_in_titles] if index_in_titles < len(titles) else None
    if true_title:
        ranking = similarities[0].argsort(descending=True)
        title_rank = ranking.tolist().index(index_in_titles) + 1
    else:
        title_rank = None

    output = f"""
Text: \n {text if text else 'No text provided'}\n\n
True title: {true_title}, score: {similarities[0, index_in_titles]:.4f}, title rank: {title_rank if title_rank else "n/a"}\n\n
Top {k} titles for text:
"""
    for i, index in enumerate(best_indices):
        output += f"{i + 1}. {titles[index]} (similarity: {similarities[0, index].item():.4f})\n"

    if print_result:
        print(output)

    return best_indices, similarities.cpu().numpy()[0, best_indices], output


def find_best_text(index_in_titles: int, k=5, print_result: bool = True):
    title_embedding = title_embeddings[index_in_titles]
    similarities: torch.Tensor = model.similarity(title_embedding, text_embeddings)

    best_indices = similarities.topk(k=k, largest=True).indices.cpu().numpy().astype(int).flatten()

    output = None

    title = titles[index_in_titles] if index_in_titles < len(texts) else None
    true_text = texts[index_in_titles] if index_in_titles < len(titles) else None
    if true_text:
        ranking = similarities[0].argsort(descending=True)
        text_rank = ranking.tolist().index(index_in_titles) + 1
    else:
        text_rank = None

    output = f"""
Title: \n {title}\n\n
True text: {true_text},\n\n Score: {similarities[0, index_in_titles]:.4f}, text rank: {text_rank if text_rank else "n/a"}\n\n
Top {k} texts matching title:
"""
    for i, index in enumerate(best_indices):
        output += f"{i + 1}. {truncate_text(texts[index], 100)} (similarity: {similarities[0, index].item():.4f})\n\n"

    if print_result:
        print(output)

    return best_indices, similarities.cpu().numpy()[0, best_indices], output


def find_best_texts_by_query(query: str, k=5, print_result: bool = True):
    query_embedding = model.encode(query, convert_to_tensor=True)
    similarities: torch.Tensor = model.similarity(query_embedding, text_embeddings)

    best_indices = similarities.topk(k=k, largest=True).indices.cpu().numpy().astype(int).flatten()

    output = None

    output = f"""Query: {query}\n\n
Top {k} texts matching query:
"""
    for i, index in enumerate(best_indices):
        output += f"{i + 1}. {truncate_text(texts[index], 100)} (similarity: {similarities[0, index].item():.4f})\n\n"

    if print_result:
        print(output)

    return best_indices, similarities.cpu().numpy()[0, best_indices], output


title_text_pairs

,page_id,page_title,section_title,text
0,31208,Ірпінь,Економіка та промисловість,У вересні 2016 року Ірпінському регіоні діяли ...
1,41207,Віскі,Виробничий процес,Виробничий процес складається з наступних осно...
2,3886,Буддизм,Праджня (мудрість): медитація віпасана,"Праджня означає мудрість, що базується на усві..."
3,25937,Мен (штат),Економіка,"Виробництво: целюлозно-паперова промисловість,..."
4,7353,Малаві,Малаві,Респу́бліка Мала́ві (до 1964 Нья́саленд) — кра...
...,...,...,...,...
9985,19354,Авреліан,Вбивство,У 275 році Авреліан на чолі великої армії руши...
9986,54195,Шахтар (Донецьк),Шахтар (Донецьк),«Шахта́р» — український футбольний клуб з міст...
9987,7790,Авокадо,Отруйність для тварин,"У низці джерел зазначено, що листя дерева авок..."
9988,28505,Рок-музика,Зародження альтернативної музичної культури,Хоча почини Velvet Underground в першій полови...


In [6]:
model = SentenceTransformer(
    # "all-MiniLM-L6-v2"
    # "all-distilroberta-v1"
    "intfloat/multilingual-e5-base"
    # "m-rudko-pn/e5-base-ukr-wikipedia"
)

In [7]:
title_embeddings = model.encode(titles, convert_to_tensor=True)
text_embeddings = model.encode(texts, convert_to_tensor=True)

In [8]:
index_slider = w.IntSlider(
    value=0,
    min=0,
    max=len(titles) - 1,
    description='Index in titles:',
    continuous_update=False,
)

text_display = w.Textarea(
    value='',
    description='Text:',
    layout=w.Layout(width='100%'),
)


def update_text_display(change):
    index = change['new']
    best_indices, similarities, output = find_best_text(index, k=5, print_result=True)
    text_display.value = output


index_slider.observe(update_text_display, names='value')

w.VBox([
    index_slider,
    text_display])

In [9]:
find_best_texts_by_query("світлова частинка", k=5, print_result=True)

Query: світлова частинка


Top 5 texts matching query:
1. Реальний масштаб (similarity: 0.8337)

2. * За одну секунду: :* світло подолає у вакуумі; :* на сітківку ока потрапить 550 трлн періодів світл... (similarity: 0.8307)

3. * Інститут електронної фізики НАН України (similarity: 0.8299)

4. Глюо́н (від glue — клей) — електрично нейтральна елементарна частинка, яка відіграє таку ж роль у си... (similarity: 0.8299)

5. Відрізок \ (similarity: 0.8261)




(array([3047, 4742,  905, 7794, 8168]),
 array([0.8336573 , 0.830665  , 0.8299349 , 0.82985854, 0.8260834 ],
       dtype=float32),
 'Query: світлова частинка\n\n\nTop 5 texts matching query:\n1. Реальний масштаб (similarity: 0.8337)\n\n2. * За одну секунду: :* світло подолає у вакуумі; :* на сітківку ока потрапить 550 трлн періодів світл... (similarity: 0.8307)\n\n3. * Інститут електронної фізики НАН України (similarity: 0.8299)\n\n4. Глюо́н (від glue — клей) — електрично нейтральна елементарна частинка, яка відіграє таку ж роль у си... (similarity: 0.8299)\n\n5. Відрізок \\ (similarity: 0.8261)\n\n')